In [1]:
import json
import pandas as pd


In [36]:
categories = "Financial,Employment,Education,Healthcare,MentalHealth,Social,Living,Smoke,SubstanceUse,Trauma,Insurance,Adherence,Literacy,Recommendation,Concern,Transportation".split(',')

In [38]:
def kw_class(kw):
    if "Financial" in kw:
        return "Financial"
    elif "Employ" in kw:
        return "Employment"
    elif "Education" in kw:
        return "Education"
    elif "Health" in kw:
        return "Healthcare"
    elif "Mental" in kw:
        return "MentalHealth"
    elif "Social" in kw:
        return "Social"
    elif "Living" in kw or "Resident" in kw:
        return "Living"
    elif "Smoke" in kw:
        return "Smoke"
    elif "Substance" in kw:
        return "SubstanceUse"
    elif "Trauma" in kw:
        return "Trauma"
    elif "Insurance" in kw:
        return "Insurance"
    elif "Adherence" in kw:
        return "Adherence"
    elif "Literacy" in kw:
        return "Literacy"
    elif "Recommendation" in kw:
        return "Recommendation"
    elif "Concern" in kw:
        return "Concern"
    elif "Transportation" in kw:
        return "Transportation"

In [47]:
output_dict = {}
for i in lv2:
    kws = list(i.keys())[3:]
    # Remove the first three keys
    if "Experiencer" not in kws:
        experiencer = 'patients'
    tmp_dict = {}
    for kw in kws:
        if kw == 'Experiencer':
            experiencer = i[kw]
        else:
            tmp_dict[kw] = i[kw]
    if experiencer not in output_dict.keys():
        output_dict[experiencer] = {}
    for k,v in tmp_dict.items():
        cat = kw_class(k)
        if cat not in output_dict[experiencer].keys():
            output_dict[experiencer][cat] = {}
        if k in output_dict[experiencer][cat].keys():
            output_dict[experiencer][cat][k].append(v)
        else:
            output_dict[experiencer][cat][k] = [v]
        

In [69]:
import pandas as pd
from collections import Counter
data = {1:output_dict}
# 1. Define the Mapping: { JSON_Category: { JSON_Key: Target_Column } }
# This tells the script where to find data for your specific column names.
column_mapping = {
    'Financial':    {'FinancialStatus': 'fin_sta'},
    'Employment':   {'EmploymentStatus': 'emp_sta'},
    'Education':    {'EducationStatus': 'edu_sta', 'EducationType': 'edu_lvl'},
    'Healthcare':   {'HealthcareType': 'hc_typ', 'MentalHealthStatus': 'mh_sta'},
    'Social':       {'SocialType': 'soc_sta', 'SocialActivity': 'soc_lvl'},
    'Living':       {'LivingStatus': 'liv_sta', 'LivingType': 'liv_typ', 'ResidentType': 'res_typ'},
    'Smoke':        {'SmokeStatus': 'smok_sta'},
    'Substance':    {'SubstanceStatus': 'subs_sta'}, # Note: Not in your JSON example, but mapped for completeness
    'Trauma':       {'TraumaType': 'trau_typ', 'TraumaStatus': 'trau_sta'},
    'Insurance':    {'InsuranceType': 'ins_typ'},
    'Adherence':    {'AdherenceType': 'adh_typ', 'AdherenceLevel': 'adh_lvl'},
    'Literacy':     {'LiteracyType': 'lit_typ', 'LiteracyLevel': 'lit_lvl'},
    'Recommendation': {'RecommendationType': 'rec_tye'},
    'Concern':      {'ConcernLevel': 'con_lvl'},
    'Transportation': {'TransportationType': 'transp_typ', 'TransportationConvenienceLevel': 'transp_lvl'}
}
final_columns = [
    'patient_id', 'experiencer', 
    'fin_sta', 'emp_sta', 'edu_sta', 'edu_lvl', 'hc_typ', 'mh_sta', 
    'soc_sta', 'soc_lvl', 'liv_sta', 'liv_typ', 'res_typ', 'smok_sta', 'subs_sta', 
    'trau_typ', 'trau_sta', 'ins_typ', 'adh_typ', 'adh_lvl', 'lit_typ', 'lit_lvl', 
    'rec_tye', 'con_lvl', 'transp_typ', 'transp_lvl'
]
# 2. Define Logic Rules
# Columns that should keep ALL values (List format) instead of picking the majority
list_columns = [
    'trau_typ', 'trau_sta', 
    'lit_typ', 'lit_lvl', 
    'rec_tye'
]

# 4. Processing Loop
rows = []

# Outer loop: Iterate through Patient IDs
for p_id, p_data in data.items():
    
    # Inner loop: Iterate through Experiencers (patients, parents, etc.)
    for experiencer, categories in p_data.items():
        row_data = {col: None for col in final_columns}
        
        # Set Identifiers
        row_data['patient_id'] = p_id
        row_data['experiencer'] = experiencer
        
        # Data Extraction
        for json_cat, field_map in column_mapping.items():
            if json_cat in categories:
                for json_key, target_col in field_map.items():
                    if json_key in categories[json_cat]:
                        values = categories[json_cat][json_key]
                        
                        if not values: continue

                        if target_col in list_columns:
                            # LIST METHOD
                            row_data[target_col] = values
                        else:
                            # MODE METHOD
                            most_common = Counter(values).most_common(1)[0][0]
                            row_data[target_col] = most_common

        rows.append(row_data)

# 5. Create DataFrame
df = pd.DataFrame(rows, columns=final_columns)
df = df[final_columns] # Enforce order


In [72]:
df.to_csv("../data/sample_tabular_sdoh.csv")